# Etapa 1 en Colab: descarga de PopResume + PaddleOCR (GPU)

Este notebook corre las dos etapas pesadas de datos en una sola sesión:

1. `src/download_data.py`: muestreo estratificado (800 CVs sin foto por ocupación, seed 42)
2. `src/ocr_extraction.py`: PaddleOCR (PP-OCRv5 server) sobre las 4,000 imágenes

**Antes de empezar**
- *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU***.
- En el panel izquierdo, ícono de la llave 🔑 (*Secretos*), crea un secreto llamado **`HF_TOKEN`** con tu token de
  Hugging Face (tipo *Read*) y activa *Acceso del notebook*.

Todo se guarda en Google Drive (`MyDrive/cv_experience_ocr_bert_data/`), así que si la sesión se cae basta con
volver a ejecutar todas las celdas: la descarga y el OCR continúan donde quedaron.

In [ ]:
!nvidia-smi

## 1. Google Drive y repositorio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/cv_experience_ocr_bert_data'
REPO_URL = 'https://github.com/JOrtega0210/cv_experience_ocr_bert.git'
REPO_DIR = '/content/cv_experience_ocr_bert'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -3

## 2. Instalación

PaddlePaddle GPU se instala primero desde el índice oficial (build CUDA 12.6) y luego PaddleOCR 3.x.
Las versiones son las mismas que `requirements.txt`.

In [ ]:
!pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr==3.7.0 datasets==5.1.0 huggingface_hub==2.3.0

In [ ]:
import os
# paddlepaddle-gpu reemplaza la librería NCCL que usa el torch preinstalado de Colab y `import torch` falla.
# Este notebook no usa torch: USE_TORCH=0 evita que `datasets` intente importarlo (lo heredan los `!python`).
os.environ['USE_TORCH'] = '0'

import paddle
print(paddle.__version__, '| CUDA:', paddle.device.is_compiled_with_cuda())
paddle.utils.run_check()

## 3. Login en Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

## 4. Persistencia en Drive

- `data/ocr/` apunta a Drive: cada JSON queda guardado apenas se procesa (permite retomar).
- Las imágenes se descargan al disco local de Colab (más rápido) y al final se comprimen en `images.zip` en Drive.
  Si ese zip ya existe, se descomprime en vez de volver a descargar.

In [ ]:
import os, shutil
os.makedirs('data/raw/images', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/ocr', exist_ok=True)
if not os.path.islink('data/ocr'):
    shutil.rmtree('data/ocr', ignore_errors=True)
    os.symlink(f'{DRIVE_DIR}/ocr', 'data/ocr')

if os.path.exists(f'{DRIVE_DIR}/images.zip') and not os.path.exists('data/raw/images/.restored'):
    !unzip -q -o {DRIVE_DIR}/images.zip -d data/raw/
    !touch data/raw/images/.restored
for f in ['metadata.csv', 'reference_text.parquet']:
    if os.path.exists(f'{DRIVE_DIR}/{f}'):
        shutil.copy(f'{DRIVE_DIR}/{f}', f'data/raw/{f}')
if os.path.exists(f'{DRIVE_DIR}/sampled_ids.csv'):
    shutil.copy(f'{DRIVE_DIR}/sampled_ids.csv', 'data/sampled_ids.csv')
print('Imágenes disponibles:', len([p for p in os.listdir('data/raw/images') if p.endswith('.png')]))

## 5. Descarga estratificada (≈ 15–30 min)

Solo corre si todavía no hay 4,000 imágenes. Al terminar copia imágenes, metadata y texto de referencia a Drive.

In [ ]:
n_images = len([p for p in os.listdir('data/raw/images') if p.endswith('.png')])
if n_images < 4000:
    !python -m src.download_data
    assert os.path.exists('data/raw/metadata.csv'), 'La descarga falló: revisa el error de arriba'
    !cd data/raw && zip -q -r -0 {DRIVE_DIR}/images.zip images -x 'images/.restored'
    for f in ['metadata.csv', 'reference_text.parquet']:
        shutil.copy(f'data/raw/{f}', f'{DRIVE_DIR}/{f}')
    shutil.copy('data/sampled_ids.csv', f'{DRIVE_DIR}/sampled_ids.csv')
else:
    print('Ya hay', n_images, 'imágenes: se omite la descarga')
!cat output/tables/target_distribution.csv

## 6. Comparación de modelos de reconocimiento en 50 CVs

Para justificar la elección del modelo: PP-OCRv5 *mobile* vs *server* sobre los mismos 50 CVs, en la misma GPU.
El CER/WER de ambos se calcula después en `src/ocr_quality.py`.

In [ ]:
!python -m src.ocr_extraction --device gpu:0 --limit 50 --out-dir data/ocr_mobile_sample \
    --det-model PP-OCRv5_mobile_det --rec-model PP-OCRv5_mobile_rec
!python -m src.ocr_extraction --device gpu:0 --limit 50 --out-dir data/ocr_server_sample
!cat output/metrics/ocr_summary_ocr_mobile_sample.json | head -12
!cat output/metrics/ocr_summary_ocr_server_sample.json | head -12

## 7. OCR completo con PP-OCRv5 server (≈ 1 h en T4)

Si la sesión se desconecta, vuelve a ejecutar todo: los CVs que ya tienen JSON en Drive se saltan.

In [ ]:
!python -m src.ocr_extraction --device gpu:0
!cat output/metrics/ocr_summary.json

## 8. Empaquetar resultados

Genera `results_ocr.zip` en Drive. Descárgalo y descomprímelo **en la raíz del repositorio local**
(`cv_experience_ocr_bert/`): crea/reemplaza `data/ocr/`, `data/raw/metadata.csv`, `data/raw/reference_text.parquet`,
`data/sampled_ids.csv` y `output/`. Opcionalmente descarga también `images.zip` y descomprímelo en `data/raw/`
(se usa en la app y en el análisis de errores).

In [ ]:
!rm -f {DRIVE_DIR}/results_ocr.zip
!zip -q -r {DRIVE_DIR}/results_ocr.zip data/ocr data/ocr_mobile_sample data/ocr_server_sample \
    data/sampled_ids.csv data/raw/metadata.csv data/raw/reference_text.parquet output
!ls -lh {DRIVE_DIR}